# ASCIIBench — Pix2Struct Fine-tuning (Colab)

Runs `scripts/run_pix2struct_finetuning.py` from the ASCIIBench repo on a free Colab GPU.
No AWS GPU quota needed — this is the fallback path while the EC2 quota request is pending.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (T4).

In [ ]:
# Confirm a GPU is attached
!nvidia-smi

In [ ]:
# Clone the repo (already includes final_dataset.jsonl)
!git clone https://github.com/KerryLuo/ASCIIBench.git
%cd ASCIIBench

In [ ]:
# Install dependencies (torch/PIL already present in Colab; this fills in the rest)
!pip install -q -r requirements-pix2struct.txt

## Persist checkpoints across sessions

Colab runtimes are ephemeral — anything not saved to Drive/S3 is lost when the
session disconnects. Pick ONE of the two options below (or both).

In [ ]:
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/ASCIIBench_pix2struct_checkpoints"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Option B: S3 (reuses the bucket + credentials from the AWS setup)
# Skip this cell if you're only using Google Drive above.
from getpass import getpass
import os

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS Access Key ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS Secret Access Key: ")
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"

S3_BUCKET = "kerryluo-pix2struct-checkpoints"  # created earlier via AWS CLI

## (Optional) Weights & Biases logging

In [ ]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted

## Run training

Adjust `--epochs` / `--batch-size` / `--lr` as needed. Colab's free T4 has less
VRAM than a g5.xlarge, so `--batch-size 2` and/or a lower `--max-patches` is a
safer starting point than the script's defaults.

In [ ]:
!python scripts/run_pix2struct_finetuning.py \
    --epochs 3 \
    --batch-size 2 \
    --checkpoint-dir "/content/drive/MyDrive/ASCIIBench_pix2struct_checkpoints" \
    --s3-bucket kerryluo-pix2struct-checkpoints

## Download checkpoints locally (optional)

If you skipped Drive/S3, or just want a local copy on top of those, zip and
download the checkpoints directory directly.

In [ ]:
from google.colab import files

!zip -r checkpoints.zip checkpoints/
files.download("checkpoints.zip")